# 07 — Matriz de predictores (`data/06_features/`)

Una fila por par `t`→`t+h` de `data/05_trayectoria_distancia/` (sin filtrar), con predictores medidos solo
en `t` o antes, para las dos eras. Lleva además, fuera de X, la clave del par, los filtros (`par_valido_c`,
`identidad_confirmada`, marcas de corte) y los pesos. El objetivo no se copia: se une a 05 por la clave.

Definiciones y roles de cada feature en `src/features.py` (`CATALOGO_FEATURES`); categorizaciones de foto
en `src/categorias_hogar.py`; contexto del aglomerado en `src/contexto_aglomerado.py`.

`TRIMESTRES` acota los orígenes a procesar. Los agregados sobre el universo de trimestres se escriben con
sufijo `_parcial` si la corrida es parcial.

In [ ]:
import sys
import time
from functools import lru_cache
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import categorias_hogar as ch
import contexto_aglomerado as cx
import cortes_trayectoria as ct
import features as ft
import panel
import pipeline_meta as pm
from armonizacion import es_historico, trimestres_publicados
from tipado_nucleo import parsear_monetario

TRIMESTRES: list[tuple[int, int]] | None = None
REHACER: bool = False

NUCLEO = REPO / "data" / "02_nucleo"
ARMONIZADO = REPO / "data" / "01_armonizado"
DISTANCIA = REPO / "data" / "02b_distancia_linea_pobreza" / "hogar"
PANEL_DIR = REPO / "data" / "03_panel"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
FEATURES = REPO / "data" / "06_features"
META = REPO / "data" / "meta"
COMMIT_GIT = pm.obtener_commit_git(REPO)

PUBLICADOS = trimestres_publicados()
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
SUFIJO = "_parcial" if ES_CORRIDA_PARCIAL else ""
trimestres_a_procesar = TRIMESTRES if ES_CORRIDA_PARCIAL else PUBLICADOS
ERA_POR_TRIMESTRE = {q: "historica" if es_historico(*q) else "regional_2016" for q in PUBLICADOS}
t0_proceso = time.time()
print(f"{len(trimestres_a_procesar)} orígenes a procesar" + (" -- CORRIDA PARCIAL" if ES_CORRIDA_PARCIAL else ""))

## Marcas de corte

Las particiones regulares de 05 no traen las marcas de `cortes_trayectoria` (las históricas sí). Se
completan acá con el mismo umbral de `ITF == 0` que usó `06c`, calculado sobre todas las particiones de 02b.

In [ ]:
pct_itf0 = {}
for p in sorted(DISTANCIA.glob("ANO4=*/TRIMESTRE=*/data.parquet")):
    anio, trimestre = int(p.parent.parent.name.split("=")[1]), int(p.parent.name.split("=")[1])
    pct_itf0[(anio, trimestre)] = ct.pct_itf_cero_declarado(pd.read_parquet(p, columns=["ITF", "ingreso_no_declarado"]))
TRIMESTRES_ITF0 = ct.trimestres_itf0_elevado(pct_itf0, ct.UMBRAL_ITF0_PCT)
print(f"trimestres con ITF==0 elevado (> {ct.UMBRAL_ITF0_PCT}%): {TRIMESTRES_ITF0}")

## Lectura de capas

Cargadores con caché por trimestre: el armado de un origen `t` lee también `t-1`, `t-3` y `t-4` (par previo,
eventos rezagados, año móvil del contexto). La base individual junta `02_nucleo` con las columnas que solo
están en `01_armonizado` (montos de `P47T`, `TOT_P12`, `PP08J*`, educación, inactividad y ocupación).

In [ ]:
# Un origen usa t, t-1, t-3 y t-4: con 16 trimestres en caché alcanza para recorrer la
# serie en orden sin tener las 86 bases individuales en memoria a la vez.
TRIMESTRES_EN_MEMORIA = 16
EXTRA_INDIVIDUAL = ["TOT_P12", "PP08J1", "PP08J2", "PP08J3", "P47T", "NIVEL_ED", "CH10", "CAT_INAC", "PP04D_COD"]
MONTOS_EXTRA = ["TOT_P12", "PP08J1", "PP08J2", "PP08J3", "P47T"]


def _leer(ruta: Path, columnas: list[str] | None = None) -> pd.DataFrame | None:
    if not ruta.exists():
        return None
    if columnas is None:
        return pd.read_parquet(ruta)
    disponibles = set(pq.read_schema(ruta).names)
    return pd.read_parquet(ruta, columns=[c for c in columnas if c in disponibles])


@lru_cache(maxsize=TRIMESTRES_EN_MEMORIA)
def hogar_en(anio: int, trimestre: int) -> pd.DataFrame | None:
    return _leer(pm.ruta_particion(NUCLEO, "hogar", anio, trimestre), ch.COLUMNAS_HOGAR)


@lru_cache(maxsize=TRIMESTRES_EN_MEMORIA)
def individual_en(anio: int, trimestre: int) -> pd.DataFrame | None:
    nucleo = _leer(pm.ruta_particion(NUCLEO, "individual", anio, trimestre))
    if nucleo is None:
        return None
    extra = _leer(pm.ruta_particion(ARMONIZADO, "individual", anio, trimestre),
                  ["CODUSU", "NRO_HOGAR", "COMPONENTE", *EXTRA_INDIVIDUAL])
    for c in MONTOS_EXTRA:
        extra[c] = parsear_monetario(extra[c])
    for c in ("NIVEL_ED", "CH10", "CAT_INAC", "PP04D_COD"):
        extra[c] = pd.to_numeric(extra[c], errors="coerce")
    return nucleo.merge(extra, on=["CODUSU", "NRO_HOGAR", "COMPONENTE"], how="left", validate="one_to_one")


@lru_cache(maxsize=TRIMESTRES_EN_MEMORIA)
def distancia_en(anio: int, trimestre: int) -> pd.DataFrame | None:
    return _leer(DISTANCIA / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet")


@lru_cache(maxsize=TRIMESTRES_EN_MEMORIA)
def pares_05(h: int, anio: int, trimestre: int) -> pd.DataFrame | None:
    pares = _leer(pm.ruta_particion(TRAYECTORIA, f"pares_h{h}", anio, trimestre))
    return None if pares is None else ft.armonizar_pares(pares, h, TRIMESTRES_ITF0)


@lru_cache(maxsize=TRIMESTRES_EN_MEMORIA)
def panel_03(h: int, anio: int, trimestre: int) -> pd.DataFrame | None:
    return _leer(pm.ruta_particion(PANEL_DIR, f"pares_h{h}", anio, trimestre))


def _concatenar(partes: list[pd.DataFrame | None]) -> pd.DataFrame:
    partes = [p for p in partes if p is not None and len(p)]
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

## Contexto del aglomerado (año móvil `t-3..t`)

Tasas de pobreza, indigencia (02b, variante A, `peso_hogar_era`), informalidad asalariada y desocupación
(personas, `PONDERA`), y Gini del IPCF relativo con peso de personas. Mínimo de 400 hogares en la ventana;
`NA` si la ventana cruza de era.

In [ ]:
@lru_cache(maxsize=None)
def agregados_en(anio: int, trimestre: int) -> pd.DataFrame:
    return cx.agregados_aglomerado_trimestre(distancia_en(anio, trimestre), individual_en(anio, trimestre))


@lru_cache(maxsize=None)
def tabla_gini_en(anio: int, trimestre: int) -> pd.DataFrame:
    return cx.tabla_hogares_gini(hogar_en(anio, trimestre), individual_en(anio, trimestre), es_historico(anio, trimestre))


DISPONIBLES_DISTANCIA = [q for q in PUBLICADOS if (DISTANCIA / f"ANO4={q[0]}" / f"TRIMESTRE={q[1]}" / "data.parquet").exists()]


def contexto_en(anio: int, trimestre: int) -> pd.DataFrame:
    ventana = cx.ventana_anio_movil((anio, trimestre), DISPONIBLES_DISTANCIA)
    tasas = cx.tasas_anio_movil({q: agregados_en(*q) for q in ventana}, ERA_POR_TRIMESTRE, (anio, trimestre))
    gini = cx.gini_anio_movil({q: tabla_gini_en(*q) for q in ventana}, ERA_POR_TRIMESTRE, (anio, trimestre))
    return tasas.merge(gini, on="AGLOMERADO", how="outer").assign(anio=anio, trimestre=trimestre)

## Armado por origen y horizonte

Para cada origen `t` y cada `h`: pares de 05 de `t`, foto del hogar en `t`, pares de 05 y de 03 que terminan
en `t` (h=1 desde `t-1`, h=4 desde `t-4`) y el par h=1 de `t-4` (previa de la 3ª visita). Se escribe una
partición por `(h, t)` con las mismas filas y clave que 05; con `REHACER=False` se reusa la partición si no
está desactualizada respecto de 05.

In [ ]:
particiones: dict[tuple[int, int, int], Path] = {}
contexto_por_trimestre: dict[tuple[int, int], pd.DataFrame] = {}
filas_resumen, n_filas_escritas, n_particiones = [], 0, 0
for anio, trimestre in trimestres_a_procesar:
    menos_1, menos_4 = panel.sumar_trimestres(anio, trimestre, -1), panel.sumar_trimestres(anio, trimestre, -4)
    for h in (1, 4):
        origen = pm.ruta_particion(TRAYECTORIA, f"pares_h{h}", anio, trimestre)
        destino = pm.ruta_particion(FEATURES, f"pares_h{h}", anio, trimestre)
        if not origen.exists():
            continue
        if not REHACER and not pm.particion_desactualizada(origen, destino):
            particiones[(h, anio, trimestre)] = destino
            continue
        if (anio, trimestre) not in contexto_por_trimestre:
            contexto_por_trimestre[(anio, trimestre)] = contexto_en(anio, trimestre)
        insumos = ft.InsumosTrimestre(
            anio=anio, trimestre=trimestre, h=h, pares=pares_05(h, anio, trimestre),
            distancia=distancia_en(anio, trimestre), hogar=hogar_en(anio, trimestre),
            individual=individual_en(anio, trimestre),
            pares_previos=_concatenar([pares_05(1, *menos_1), pares_05(4, *menos_4), pares_05(1, *menos_4)]),
            panel_previos=_concatenar([panel_03(1, *menos_1), panel_03(4, *menos_4)]),
            individual_previo={q: individual_en(*q) for q in (menos_1, menos_4) if individual_en(*q) is not None},
            contexto_aglomerado=contexto_por_trimestre[(anio, trimestre)],
        )
        features = ft.construir_features_trimestre(insumos)
        clave_05 = pd.read_parquet(origen, columns=ft.CLAVE_PAR)
        cruce = clave_05.merge(features[ft.CLAVE_PAR], on=ft.CLAVE_PAR, how="outer", indicator=True)
        assert (cruce["_merge"] == "both").all() and len(cruce) == len(clave_05), f"{anio}T{trimestre} h={h}: clave distinta de 05"
        destino.parent.mkdir(parents=True, exist_ok=True)
        features.to_parquet(destino, index=False)
        particiones[(h, anio, trimestre)] = destino
        n_filas_escritas += len(features)
        n_particiones += 1
        pm.actualizar_resumen_trimestre(META / "resumen_06_features.csv", [{
            "anio": anio, "trimestre": trimestre, "h": h, "era": ERA_POR_TRIMESTRE[(anio, trimestre)],
            "n_pares": len(features), "n_par_valido_c": int((features["par_valido_c"] == True).sum()),
            "n_columnas": features.shape[1], "n_features_x": ft.construir_matriz_X(features).shape[1]}],
            claves=("anio", "trimestre", "h"))
        print(f"{anio}T{trimestre} h={h}: {len(features):,} pares, {features.shape[1]} columnas")

## Anti-join con 05 en todas las particiones

Se relee la clave de cada partición de 05 y de 06 desde disco (también las reusadas): las dos tienen que
tener exactamente los mismos pares.

In [ ]:
filas_antijoin = []
for h in (1, 4):
    for anio, trimestre in trimestres_a_procesar:
        origen = pm.ruta_particion(TRAYECTORIA, f"pares_h{h}", anio, trimestre)
        if not origen.exists():
            continue
        destino = pm.ruta_particion(FEATURES, f"pares_h{h}", anio, trimestre)
        clave_05 = pd.read_parquet(origen, columns=ft.CLAVE_PAR)
        clave_06 = pd.read_parquet(destino, columns=ft.CLAVE_PAR) if destino.exists() else clave_05.iloc[0:0]
        cruce = clave_05.merge(clave_06, on=ft.CLAVE_PAR, how="outer", indicator=True)
        filas_antijoin.append({"h": h, "anio": anio, "trimestre": trimestre, "n_05": len(clave_05), "n_06": len(clave_06),
                               "solo_05": int((cruce["_merge"] == "left_only").sum()),
                               "solo_06": int((cruce["_merge"] == "right_only").sum())})
antijoin = pd.DataFrame(filas_antijoin)
print(f"{len(antijoin)} particiones ({antijoin[['anio', 'trimestre']].drop_duplicates().shape[0]} orígenes): "
      f"{antijoin['n_05'].sum():,} pares en 05, {antijoin['n_06'].sum():,} en 06, "
      f"solo en 05: {antijoin['solo_05'].sum()}, solo en 06: {antijoin['solo_06'].sum()}")
display(antijoin.groupby("h")[["n_05", "n_06", "solo_05", "solo_06"]].sum())

## Faltantes por feature

`pct_na_estructural_*` cuenta las filas donde la feature no aplica por diseño (1ª visita sin par previo,
hogar sin activos, `ITF <= 0`); `pct_na_dato_*` es el NA entre las filas donde sí aplica. Se reporta en todos
los pares y en el universo `par_valido_c`.

In [ ]:
reportes = []
for (h, anio, trimestre), ruta in sorted(particiones.items()):
    features = pd.read_parquet(ruta)
    reportes.append(ft.reporte_na(features).assign(h=h, anio=anio, trimestre=trimestre, era=ERA_POR_TRIMESTRE[(anio, trimestre)]))
reporte_na = pd.concat(reportes, ignore_index=True)
columnas_id = ["h", "anio", "trimestre", "era"]
reporte_na = reporte_na[columnas_id + [c for c in reporte_na.columns if c not in columnas_id]]
reporte_na.to_csv(META / f"features_na_trimestre{SUFIJO}.csv", index=False)

con_na = reporte_na[(reporte_na["pct_na_par_valido_c"] > 0) & reporte_na["en_x"]]
resumen_era = con_na.groupby(["era", "h", "bloque", "feature"])[
    ["pct_na_estructural_par_valido_c", "pct_na_dato_par_valido_c"]].agg(["mean", "min", "max"]).round(2)
display(resumen_era)
display(con_na.pivot_table(index=["bloque", "feature"], columns="anio", values="pct_na_dato_par_valido_c",
                           aggfunc="mean").round(1))

motivos = pd.concat([pd.read_parquet(ruta, columns=["motivo_previa_na_t", "motivo_previa_na_detalle_t", "par_valido_c"])
                     .assign(era=ERA_POR_TRIMESTRE[(a, t)], h=h) for (h, a, t), ruta in particiones.items()], ignore_index=True)
motivos = motivos[motivos["par_valido_c"] == True]  # noqa: E712
for columna in ("motivo_previa_na_t", "motivo_previa_na_detalle_t"):
    display((100 * pd.crosstab([motivos["era"], motivos["h"]], motivos[columna].fillna("NA"), normalize="index")).round(2))

## Validación del Gini contra `data/meta/gini_aglomerado.csv`

`gini_aglomerado.csv` lo escribió `05_composicion_quintiles.ipynb` (commit `994a4b5`, corrida del
2026-10-03 23:19 en `corridas.csv`); ese notebook ya no está en el repo. `contexto_aglomerado.tabla_hogares_gini`
replica su receta, así que el Gini de año móvil se valida contra la columna `gini_anio_movil` (±1e-4: el CSV
está redondeado a 4 decimales).

In [ ]:
referencia = pd.read_csv(META / "gini_aglomerado.csv")[["anio", "trimestre", "aglomerado", "gini_anio_movil"]]
contexto = pd.concat(contexto_por_trimestre.values(), ignore_index=True) if contexto_por_trimestre else pd.DataFrame()
if len(contexto):
    contexto = contexto.merge(referencia.rename(columns={"aglomerado": "AGLOMERADO", "gini_anio_movil": "gini_referencia"}),
                              on=["anio", "trimestre", "AGLOMERADO"], how="left")
    contexto["diferencia_gini"] = contexto["gini_ipcf_aglo"] - contexto["gini_referencia"]
    contexto.to_csv(META / f"contexto_aglomerado_trimestre{SUFIJO}.csv", index=False)
    ambos = contexto[contexto["gini_ipcf_aglo"].notna() & contexto["gini_referencia"].notna()]
    na_distinto = int((contexto["gini_ipcf_aglo"].isna() != contexto["gini_referencia"].isna()).sum())
    fuera = int((ambos["diferencia_gini"].abs() > 1e-4).sum())
    print(f"{len(ambos)} aglomerado-trimestre comparables; |dif| máx {ambos['diferencia_gini'].abs().max():.2e}; "
          f"{fuera} fuera de ±1e-4; {na_distinto} con NA en un solo lado")
    display(contexto.groupby(["anio", "trimestre"])[["tasa_pobreza_aglo", "tasa_informalidad_aglo", "gini_ipcf_aglo"]]
            .describe().T.round(3))

## `PP04D_COD`: 5º dígito (calificación) en todos los trimestres

Diagnóstico pedido antes de construir la variante D de precariedad: cuántos ocupados tienen un 5º dígito
fuera de 1–4, con el código leído como entero y completado a 5 dígitos (`calificacion_ocupacion`) y leído
sin completar. Recorre todos los trimestres publicados, no solo los de `TRIMESTRES`.

In [ ]:
filas_pp04d = []
for anio, trimestre in PUBLICADOS:
    ocupacion = _leer(pm.ruta_particion(ARMONIZADO, "individual", anio, trimestre), ["ESTADO", "CAT_OCUP", "PP04D_COD"])
    ocupados = ocupacion[pd.to_numeric(ocupacion["ESTADO"], errors="coerce") == 1]
    codigo = pd.to_numeric(ocupados["PP04D_COD"], errors="coerce")
    sin_completar = codigo.dropna().astype("int64").astype(str)
    completado = sin_completar.str.zfill(5).str[4]
    cuenta_propia = (pd.to_numeric(ocupados["CAT_OCUP"], errors="coerce") == 2).reindex(completado.index)
    filas_pp04d.append({
        "anio": anio, "trimestre": trimestre, "n_ocupados": len(ocupados),
        "pct_codigo_corto": 100 * (sin_completar.str.len() < 5).mean(),
        "pct_digito5_cero": 100 * (completado == "0").mean(),
        "pct_digito5_cero_cuenta_propia": 100 * (completado[cuenta_propia] == "0").mean(),
        "pct_digito5_fuera_1_4": 100 * (~completado.isin(list("1234"))).mean(),
        "pct_cuenta_propia_sin_calificacion": 100 * ch.calificacion_ocupacion(ocupados["PP04D_COD"])[
            (pd.to_numeric(ocupados["CAT_OCUP"], errors="coerce") == 2)].isna().mean(),
        "digitos5_presentes": "".join(sorted(completado.unique())),
        "n_codigos_distintos": codigo.nunique(),
    })
pp04d = pd.DataFrame(filas_pp04d)
pp04d.to_csv(META / "features_pp04d_calificacion.csv", index=False)
print(f"trimestres con algún 5º dígito == 0: {int((pp04d['pct_digito5_cero'] > 0).sum())} de {len(pp04d)}")
display(pp04d.groupby(pp04d["anio"] >= 2016)[["pct_codigo_corto", "pct_digito5_cero", "pct_digito5_fuera_1_4",
                                              "pct_cuenta_propia_sin_calificacion", "n_codigos_distintos"]].describe().T.round(3))

## Continuidad de las fuentes de ingreso en el corte 2023T4

Medias ponderadas (`PONDIH`) de los shares por trimestre, 2022T1–2025T4, sobre todos los hogares con
`ITF > 0` y `PONDIH > 0` (corte transversal, no solo los pares). Un salto entre 2023T3 y 2023T4 fuera de la
variación habitual entre trimestres se marca como anomalía.

In [ ]:
SHARES_CONTINUIDAD = ["share_jubilacion", "share_residuo_no_laboral", "share_jubilacion_continua", "share_transferencias",
                      "share_otros_no_laborales"]
filas_continuidad = []
for anio, trimestre in [q for q in PUBLICADOS if (2022, 1) <= q <= (2025, 4)]:
    hogar, individual = hogar_en(anio, trimestre), individual_en(anio, trimestre)
    itf = hogar.set_index(ch.CLAVE_HOGAR)["ITF"]
    shares = ch.fuentes_ingreso_hogar(individual, ch.fuentes_por_trimestre(anio, trimestre), itf).merge(
        hogar[[*ch.CLAVE_HOGAR, "PONDIH"]], on=ch.CLAVE_HOGAR)
    shares["share_jubilacion_continua"] = shares["share_jubilacion"] + shares["share_residuo_no_laboral"]
    peso = pd.to_numeric(shares["PONDIH"], errors="coerce").astype(float)
    universo = (pd.to_numeric(itf, errors="coerce").to_numpy() > 0) & (peso > 0).to_numpy()
    fila = {"anio": anio, "trimestre": trimestre, "n_hogares": int(universo.sum())}
    for share in ["share_laboral", *SHARES_CONTINUIDAD]:
        valido = universo & shares[share].notna().to_numpy()
        fila[share] = float(np.average(shares.loc[valido, share], weights=peso[valido]))
    filas_continuidad.append(fila)
continuidad = pd.DataFrame(filas_continuidad)
continuidad.to_csv(META / "features_continuidad_2023t4.csv", index=False)
display(continuidad.round(4))

saltos = continuidad[SHARES_CONTINUIDAD].diff()
corte = continuidad.index[(continuidad["anio"] == 2023) & (continuidad["trimestre"] == 4)][0]
for share in SHARES_CONTINUIDAD:
    habitual = saltos[share].drop(corte).abs().median()
    salto = saltos.loc[corte, share]
    marca = "ANOMALÍA" if abs(salto) > 3 * habitual and abs(salto) > 0.005 else "sin salto"
    print(f"{share}: 2023T3->2023T4 {salto:+.4f} (mediana |Δ| del resto {habitual:.4f}) -> {marca}")

### Estrategias V13–V17 en el corte 2023T4

Mismo universo y peso que la tabla de shares. Para cada pregunta, % de hogares con respuesta 1 (sí) y % con
un código fuera de {1, 2}; más los dos indicadores que arma `calcular_estrategias`. Siguen fuera de X hasta
que se decida con esta tabla.

In [ ]:
ESTRATEGIAS = ["V13", "V14", "V15", "V16", "V17"]
filas_estrategias = []
for anio, trimestre in [q for q in PUBLICADOS if (2022, 1) <= q <= (2025, 4)]:
    hogar = hogar_en(anio, trimestre)
    peso = pd.to_numeric(hogar["PONDIH"], errors="coerce").astype(float)
    universo = (pd.to_numeric(hogar["ITF"], errors="coerce") > 0) & (peso > 0)
    fila = {"anio": anio, "trimestre": trimestre, "n_hogares": int(universo.sum())}
    for v in ESTRATEGIAS:
        codigo = pd.to_numeric(hogar[v], errors="coerce") if v in hogar else pd.Series(np.nan, index=hogar.index)
        fila[f"pct_{v}_si"] = 100 * np.average(codigo[universo] == 1, weights=peso[universo])
        fila[f"pct_{v}_fuera_1_2"] = 100 * np.average(~codigo[universo].isin([1, 2]), weights=peso[universo])
    indicadores = ch.calcular_estrategias(hogar)
    for c in indicadores.columns:
        fila[f"pct_{c}"] = 100 * np.average(indicadores.loc[universo, c].astype(bool), weights=peso[universo])
    filas_estrategias.append(fila)
estrategias = pd.DataFrame(filas_estrategias)
estrategias.to_csv(META / "features_continuidad_estrategias_2023t4.csv", index=False)
display(estrategias.set_index(["anio", "trimestre"]).round(2))

**Anomalía documentada (2026-11-10).** `share_residuo_no_laboral` (`T_VI` menos los no laborales listados)
cae de 1,34% del ITF en 2023T3 a 0,00% en 2023T4 y queda en cero desde entonces. En el mismo corte
`share_jubilacion` sube de 22,40% a 23,50%, y la suma jubilación + residuo es casi continua (23,74% → 23,50%).
Lectura [inferido]: hasta 2023T3 los bonos previsionales no tenían columna propia y quedaban en el residuo;
el cuestionario de 2023T4 los capta dentro de `V2_0x_M`/`V21_0x_M`/`V22_0x_M`. No es un problema de la
armonización por suma (que reproduce `T_VI` al 100%), pero sí un cambio de contenido de `share_jubilacion_t`
justo al comienzo del holdout. Por eso en X entra `share_jubilacion_continua_t` (= jubilación + residuo);
`share_jubilacion_t` y `share_residuo_no_laboral_t` se calculan pero quedan fuera de la lista blanca, y
`cuestionario_2023t4_t` es solo una marca (prohibida en X: vale 0 en todo train y 1 en todo el holdout).

## Distribución de vivienda en los orígenes procesados

Proporción ponderada (`peso_hogar_era_t`) de cada componente de vivienda entre los pares h=1 de cada origen.

In [ ]:
COLUMNAS_VIVIENDA = [f.nombre for f in ft.CATALOGO_FEATURES if f.bloque == "vivienda"]
filas_vivienda = []
for (h, anio, trimestre), ruta in sorted(particiones.items()):
    if h != 1:
        continue
    features = pd.read_parquet(ruta, columns=[*COLUMNAS_VIVIENDA, "peso_hogar_era_t"])
    peso = features["peso_hogar_era_t"].astype(float)
    filas_vivienda.append({"anio": anio, "trimestre": trimestre, "era": ERA_POR_TRIMESTRE[(anio, trimestre)],
                           **{c: 100 * np.average(features[c].astype(float), weights=peso) for c in COLUMNAS_VIVIENDA}})
vivienda = pd.DataFrame(filas_vivienda)
vivienda.to_csv(META / f"vivienda_distribuciones{SUFIJO}.csv", index=False)
display(vivienda.set_index(["anio", "trimestre", "era"]).T.round(1))

## Catálogo y registro de la corrida

In [ ]:
ft.catalogo_como_tabla().to_csv(META / "features_catalogo.csv", index=False)
duracion = time.time() - t0_proceso
pm.registrar_corrida(
    META / "corridas.csv", "07_features",
    {"TRIMESTRES": trimestres_a_procesar if ES_CORRIDA_PARCIAL else None, "REHACER": REHACER,
     "trimestres_itf0_elevado": TRIMESTRES_ITF0, "min_hogares_ventana": cx.MIN_HOGARES_VENTANA},
    n_particiones, n_filas_escritas, duracion, COMMIT_GIT)
print(f"corrida registrada: {n_particiones} particiones escritas, {n_filas_escritas:,} filas, {duracion:.0f}s")